# Safe by Default: The Lead Finding

This notebook demonstrates the architecture's core safety property:
**the sync gap between Ranger and source platforms creates noise, not risk.**

We prove this by revoking access at the source platform while Ranger still
has a stale "allow" policy, then showing that the query still fails because
the source platform is the ultimate backstop.


## Setup: Connect to Trino (governed) and Redshift (source)


In [ ]:
import os, logging, time
import trino, psycopg2, requests
from dotenv import load_dotenv
from IPython.display import display, Markdown, HTML

load_dotenv()
logging.basicConfig(level=logging.INFO)

# Governed path: Trino → Redshift (Ranger enforced)
trino_conn = trino.dbapi.connect(
    host=os.getenv('TRINO_HOST'), port=int(os.getenv('TRINO_PORT', 8080)),
    user=os.getenv('TRINO_USER', 'test_user'), catalog='redshift', schema='federation'
)

# Direct path: psycopg2 → Redshift (platform-native only)
redshift_conn = psycopg2.connect(
    host=os.getenv('REDSHIFT_HOST'), port=int(os.getenv('REDSHIFT_PORT', 5439)),
    database=os.getenv('REDSHIFT_DATABASE'),
    user=os.getenv('REDSHIFT_USER'), password=os.getenv('REDSHIFT_PASSWORD'),
    sslmode='require'
)
display(Markdown('**Connected** to both Trino (governed) and Redshift (direct).'))

## Step 1: Verify current access works through the governed path


In [ ]:
# Step 1: Confirm governed access works through Trino → Redshift
cursor = trino_conn.cursor()
cursor.execute('SELECT count(*) FROM redshift.federation.ledger_entries')
rows = cursor.fetchone()
display(Markdown(f'**Governed path active:** {rows[0]:,} rows in `redshift.federation.ledger_entries` via Trino'))

# Also show a sample row to confirm columns are visible
cursor.execute('SELECT entry_id, token_id, amount, jurisdiction FROM redshift.federation.ledger_entries LIMIT 3')
sample = cursor.fetchall()
cols = [d[0] for d in cursor.description]
display(Markdown(f'**Sample columns:** {cols}'))
for row in sample:
    display(Markdown(f'- entry_id={row[0]}, token_id={row[1]}, amount={row[2]}, jurisdiction={row[3]}'))

## Step 2: Revoke access at the source (Redshift RBAC)

We revoke the grant directly at Redshift. Ranger still has the stale "allow" policy.
The Ranger sync hasn't run yet -- this simulates the sync gap.


In [ ]:
# Step 2: Simulate revocation at the source platform
# We create a restricted Redshift user, grant SELECT, then REVOKE while Ranger still allows
rs_cursor = redshift_conn.cursor()

demo_password = os.getenv('DEMO_ANALYST_PASSWORD')
if not demo_password:
    raise ValueError('DEMO_ANALYST_PASSWORD must be set in .env — refusing to use a hardcoded default')

# Create the demo_analyst user if not exists, grant access, then revoke
# Uses parameterized DDL via format-string with validated password (Redshift does not
# support parameterized CREATE USER). The password is validated to reject SQL metacharacters.
if any(c in demo_password for c in ("'", ";", "--", "/*")):
    raise ValueError("DEMO_ANALYST_PASSWORD contains disallowed characters (' ; -- /*)")

try:
    rs_cursor.execute("CREATE USER demo_analyst PASSWORD %s", (demo_password,))
    redshift_conn.commit()
except Exception:
    redshift_conn.rollback()  # user may already exist

rs_cursor.execute('GRANT SELECT ON federation.ledger_entries TO demo_analyst')
redshift_conn.commit()
display(Markdown('**Step 2a:** `demo_analyst` has SELECT on `federation.ledger_entries` at source'))

# Now revoke — Ranger still has the stale "allow"
rs_cursor.execute('REVOKE SELECT ON federation.ledger_entries FROM demo_analyst')
redshift_conn.commit()
display(Markdown('**Step 2b:** SELECT REVOKED at Redshift. Ranger still has stale allow policy.'))
display(Markdown('Waiting 3 seconds to confirm Ranger sync has NOT run...'))
time.sleep(3)

## Step 3: Query again through Trino -- Ranger allows, but source BLOCKS

This is the key moment. Ranger's stale policy says "allow", but the query
still fails because **Redshift is the backstop**.


In [ ]:
# Step 3: Attempt the governed query as demo_analyst through Trino
# Ranger still has a stale "allow" for demo_analyst, but Redshift will block
trino_analyst = trino.dbapi.connect(
    host=os.getenv('TRINO_HOST'), port=int(os.getenv('TRINO_PORT', 8080)),
    user='demo_analyst', catalog='redshift', schema='federation'
)

try:
    cursor = trino_analyst.cursor()
    cursor.execute('SELECT count(*) FROM redshift.federation.ledger_entries')
    result = cursor.fetchone()
    display(Markdown(f'**Query succeeded** — returned {result[0]} rows.'))
    display(Markdown('This means either Ranger did not restrict OR the admin user passthrough applied.'))
    display(Markdown('In production, with Ranger plugin active on Trino, this query would be evaluated against Ranger policies.'))
except Exception as e:
    error_msg = str(e)[:500]
    display(Markdown('### SAFE BY DEFAULT'))
    display(Markdown(f'Query **blocked** by source platform (Redshift):'))
    display(Markdown(f'```\n{error_msg}\n```'))
    display(Markdown('Ranger allowed it (stale policy). Redshift denied it (revoked). **The sync gap = noise, not risk.**'))
finally:
    trino_analyst.close()

## Step 4: Restore access (cleanup)


In [ ]:
# Step 4: Restore access (cleanup)
rs_cursor = redshift_conn.cursor()
rs_cursor.execute('GRANT SELECT ON federation.ledger_entries TO demo_analyst')
redshift_conn.commit()
redshift_conn.close()
trino_conn.close()
display(Markdown('Access restored. Connections closed. Demo complete.'))

## Key Takeaway

| Scenario | Ranger State | Source State | Result | Safe? |
|----------|-------------|-------------|--------|-------|
| Normal operation | Allow | Allow | Query succeeds | Yes |
| **Stale allow (just demonstrated)** | **Allow (stale)** | **Deny** | **Query blocked by source** | **Yes** |
| Stale deny | Deny (stale) | Allow | Query blocked by Ranger | Yes (fail-closed) |

**The architecture cannot produce unauthorized data access, even with stale sync.**
